# **Limpieza general de datos**
### **Carga de datos**

In [195]:
import pandas as pd

# Ruta relativa según tu proyecto
ruta_datos = '../data/data.csv'
df_raw = pd.read_csv(ruta_datos)

### **Vista inicial del DataFrame**

In [196]:
print('Vista inicial del DataFrame:')
print('------------------------------------------')
print(df_raw)

Vista inicial del DataFrame:
------------------------------------------
           country  location_id        observation_id               user_id  \
0      Afghanistan          160  edu_6447710491377664  edu_4771167334563840   
1      Afghanistan          160  edu_5214105756762112  edu_4525461102395392   
2      Afghanistan          160  edu_6358619313668096  edu_6260553241853952   
3      Afghanistan          160  edu_5023246000062464  edu_6732475616722944   
4      Afghanistan          160  edu_6525867185668096  edu_5539453241393152   
...            ...          ...                   ...                   ...   
23347        Yemen          157  edu_6642378533502976  edu_5383239220068352   
23348        Yemen          157  edu_6049497825411072  edu_5562530184560640   
23349        Yemen          157  edu_5184529160732672  edu_6481353718104064   
23350        Yemen          157  edu_6116795667972096  edu_5986846446452736   
23351        Yemen          157  edu_5354576713875456  edu_

### **Información inicial del DataFrame**

In [197]:
print('Información inicial del DataFrame:')
print('------------------------------------------')
df_raw.info()

Información inicial del DataFrame:
------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23352 entries, 0 to 23351
Columns: 122 entries, country to edu_ch4_help_99
dtypes: float64(93), int64(11), object(18)
memory usage: 21.7+ MB


### **Reestructuración del dataset**

In [198]:
# Columnas de cuidador
columnas_cuidador = [col for col in df_raw.columns if 'edu_ch' not in col]

# Columnas de niños (por grupo)
columnas_ch1 = [col for col in df_raw.columns if 'ch1' in col]
columnas_ch2 = [col for col in df_raw.columns if 'ch2' in col]
columnas_ch3 = [col for col in df_raw.columns if 'ch3' in col]
columnas_ch4 = [col for col in df_raw.columns if 'ch4' in col]

In [199]:
def crear_df_nino(df_raw, columnas_cuidador, columnas_niño, numero_niño):
    df_nino = df_raw[columnas_cuidador + columnas_niño].copy()

    # Renombrar columnas para que tengan un nombre común
    prefijo_original = f'edu_ch{numero_niño}_'
    renombrar_columnas = {col: col.replace(prefijo_original, 'ch_') for col in columnas_niño}
    df_nino.rename(columns=renombrar_columnas, inplace=True)

    # Añadir columna indicando qué número de niño era (1, 2, 3, 4)
    df_nino['child_number'] = numero_niño

    return df_nino

df_ch1 = crear_df_nino(df_raw, columnas_cuidador, columnas_ch1, 1)
df_ch2 = crear_df_nino(df_raw, columnas_cuidador, columnas_ch2, 2)
df_ch3 = crear_df_nino(df_raw, columnas_cuidador, columnas_ch3, 3)
df_ch4 = crear_df_nino(df_raw, columnas_cuidador, columnas_ch4, 4)

# Reestructurar el dataset para que cada niño tenga su propia fila
df_ninos = pd.concat([df_ch1, df_ch2, df_ch3, df_ch4], ignore_index=True)

# Eliminamos los "niños fantasma" que se crearon
df_ninos = df_ninos[df_ninos['child_number'] <= df_ninos['edu_students']]
df_ninos.reset_index(drop=True, inplace=True)

### **Vista del nuevo DataFrame**

In [200]:
print('Vista inicial del DataFrame:')
print('------------------------------------------')
print(df_ninos)

Vista inicial del DataFrame:
------------------------------------------
           country  location_id        observation_id               user_id  \
0      Afghanistan          160  edu_6447710491377664  edu_4771167334563840   
1      Afghanistan          160  edu_5214105756762112  edu_4525461102395392   
2      Afghanistan          160  edu_6358619313668096  edu_6260553241853952   
3      Afghanistan          160  edu_5023246000062464  edu_6732475616722944   
4      Afghanistan          160  edu_6525867185668096  edu_5539453241393152   
...            ...          ...                   ...                   ...   
45927        Yemen          157  edu_6640377582059520  edu_6670626596323328   
45928        Yemen          157  edu_5873449062105088  edu_5259217276764160   
45929        Yemen          157  edu_4682905065619456  edu_6393074203754496   
45930        Yemen          157  edu_6217330760876032  edu_5063276806537216   
45931        Yemen          157  edu_6116795667972096  edu_

### **Información del nuevo DataFrame**

In [201]:
print('Información inicial del DataFrame:')
print('------------------------------------------')
df_ninos.info()

Información inicial del DataFrame:
------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45932 entries, 0 to 45931
Data columns (total 42 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   country              45932 non-null  object 
 1   location_id          45932 non-null  int64  
 2   observation_id       45932 non-null  object 
 3   user_id              45932 non-null  object 
 4   submission_time      45932 non-null  object 
 5   age                  45932 non-null  int64  
 6   gender               45932 non-null  int64  
 7   geography            45932 non-null  int64  
 8   financial_situation  45932 non-null  int64  
 9   education            45932 non-null  int64  
 10  employment_status    45932 non-null  int64  
 11  ethnicity            45932 non-null  object 
 12  religion             45932 non-null  object 
 13  edu_students         45932 non-null  float64
 14  ch_gende

### **Eliminación de columnas**
Vamos a eliminar las columnas originales de elección múltiple ya que el mismo dataset contiene los mismos datos pero separados por nuevas columnas (depende de la opción que se haya elegido).

In [202]:
columnas_multiple = [col for col in df_ninos.columns 
                     if df_ninos[col].astype(str).str.contains('\^', regex=True).any()]

df_ninos.drop(columns=columnas_multiple, inplace=True)

print("Nuevas dimensiones del DataFrame:", df_ninos.shape)

<>:2: SyntaxWarning: invalid escape sequence '\^'
<>:2: SyntaxWarning: invalid escape sequence '\^'
C:\Users\andy-\AppData\Local\Temp\ipykernel_10208\2466402561.py:2: SyntaxWarning: invalid escape sequence '\^'
  if df_ninos[col].astype(str).str.contains('\^', regex=True).any()]


Nuevas dimensiones del DataFrame: (45932, 39)


### Conversión de `submission_time` a tipo `datetime`.

In [203]:
df_ninos['submission_time'] = pd.to_datetime(df_ninos['submission_time'], errors = 'coerce')

# Verificación del cambio
print('Columna submission_time transformada')
print('------------------------------------------')
print(df_ninos['submission_time'])

Columna submission_time transformada
------------------------------------------
0       2021-05-28 19:53:45.487000+00:00
1       2021-05-28 20:22:11.486000+00:00
2       2021-05-28 21:05:35.887000+00:00
3       2021-05-28 21:28:21.402000+00:00
4       2021-05-28 23:25:45.569000+00:00
                      ...               
45927   2021-06-20 02:25:58.151000+00:00
45928   2021-06-20 02:41:42.992000+00:00
45929   2021-06-20 02:08:49.794000+00:00
45930   2021-06-20 06:22:04.398000+00:00
45931   2021-06-20 02:55:07.400000+00:00
Name: submission_time, Length: 45932, dtype: datetime64[ns, UTC]


### **Columnas con valores nulos**

In [204]:
valores_nulos = df_ninos.isnull().sum()
porcentaje_nulos = (valores_nulos / len(df_ninos)) * 100

nulos_df = pd.DataFrame({
    'Valores_Nulos': valores_nulos,
    'Porcentaje_Nulos': porcentaje_nulos
}).sort_values(by='Porcentaje_Nulos', ascending=False)

print(nulos_df[nulos_df['Valores_Nulos'] > 0])

                    Valores_Nulos  Porcentaje_Nulos
ch_school_reopen            44469         96.814857
ch_no_school_why            42871         93.335801
ch_internet_access          26182         57.001655
ch_help_6                   23841         51.904990
ch_help_99                  23841         51.904990
ch_help_0                   23841         51.904990
ch_help_2                   23841         51.904990
ch_help_1                   23841         51.904990
ch_help_4                   23841         51.904990
ch_help_5                   23841         51.904990
ch_help_3                   23841         51.904990
ch_remote_method_4          23830         51.881042
ch_remote_method_5          23830         51.881042
ch_remote_method_3          23830         51.881042
ch_remote_method_2          23830         51.881042
ch_remote_method_1          23830         51.881042
ch_support_1                20916         45.536881
ch_support_0                20916         45.536881
ch_support_2

Existen columnas con demasiados valores faltantes pero no es por falta de valores (al menos no a propósito). Recordemos que algunas preguntas son condicionadas a respuestas anteriores, entonces estos valores NA se pueden categorizar como "no aplica" y escoger un modelo que maneje bien los NA.

De esta forma, no hay imputación de valores faltantes ni eliminación de las mismas.
### **Guardar dataset limpio**

In [205]:
ruta_guardado = '../data/datos_limpios.csv'
df_ninos.to_csv(ruta_guardado, index=False)

print(f"Dataset limpio guardado exitosamente en {ruta_guardado}")

Dataset limpio guardado exitosamente en ../data/datos_limpios.csv
